In [1]:
import ee
import geemap
import geopandas as gpd   
ee.Authenticate(auth_mode='localhost')
ee.Initialize(project='earth-engine-auth-project')  



### parameters 

In [4]:
### gee dataset
dset_l5 = "LANDSAT/LT05/C02/T1_L2"
dset_l7 = "LANDSAT/LE07/C02/T1_L2"
dset_l8 = "LANDSAT/LC08/C02/T1_L2"
dset_l9 = "LANDSAT/LC09/C02/T1_L2"
## true color composite bands
bands_vis_l57 = ['SR_B3', 'SR_B2', 'SR_B1']       # landsat 5,7
bands_vis_l89 = ['SR_B4', 'SR_B3', 'SR_B2']   # landsat 8,9
bands_vis_s2 = ['B4','B3','B2']  
## selected Bands (Landsat 5/7 SR)  
bands_sel_l57 = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']  
bands_sel_l89 = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'] 
bands_sel_s2 = ['B2','B3','B4','B8','B11','B12']  
s2_dset = "COPERNICUS/S2_SR_HARMONIZED"
dset_search = dset_l9
bands_vis  = bands_vis_l89 
bands_sel = bands_sel_l89


In [ ]:
path_ygp_vec = 'data/boundary/ygp_region.gpkg'
ygp_gdf = gpd.read_file(path_ygp_vec) 
ygp_region = geemap.gdf_to_ee(ygp_gdf)


### image selection

In [4]:
## region determination
region = ee.Geometry.Rectangle(107.88, 28.52, 108.53, 29.07) # (lon_min,lat_min,lon_max,lat_max)

## to make the image size > 2000x2000 pixels, 
## （1）the area may be larger than 3,600,000,000 for landsat 5789 images
## （2）larger than 400,000,000 for sentinel-2 
print('scene area:', region.area().getInfo())

# Landsat 5789 and s2 image search
start_time = '2017-06-01'
end_time = '2025-09-30'
# ----- image search and selection ----- 
## l5789
img_col = (ee.ImageCollection(dset_search)
           .filter(ee.Filter.lt('CLOUD_COVER_LAND', 10))
           .filter(ee.Filter.gt('CLOUD_COVER_LAND', 0))
           .filterBounds(region)
           .filterDate(start_time, end_time)
           .sort('CLOUD_COVER_LAND')
            )

# ## sentinel-2
# img_col = (ee.ImageCollection(dset_search)
#            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15))
#            .filter(ee.Filter.gt('CLOUDY_PIXEL_PERCENTAGE', 0))
#            .filterBounds(region)
#            .filterDate(start_time, end_time)
#            .sort('CLOUDY_PIXEL_PERCENTAGE')           
#            )

img_sel = img_col.filter(ee.Filter.contains('.geo', region))  
num_img_sel = img_sel.size().getInfo()
assert num_img_sel > 0, "No image found for the specified region and time range."

print("Total selected images:", img_sel.size().getInfo())

# # ---- specific landsat image selection ---- 
img_sel = ee.Image(img_sel.toList(img_sel.size()).get(1))
scene_sel = img_sel.clip(region)
if str(scene_sel.id().getInfo()).startswith('L'):    # if Landsat, apply scaling factors
    scene_sel = scene_sel.select('SR_B.').multiply(0.0000275).add(-0.2).multiply(10000).toInt16() ;  ## scaling for landsat-5789

img_id = dset_search + "/" + str(img_sel.id().getInfo())
print('image id for the selected image:', img_id)


scene area: 3873679402.257138
Total selected images: 2
image id for the selected image: LANDSAT/LC09/C02/T1_L2/LC09_126040_20241030


### check

In [ ]:
Map = geemap.Map()
Map.centerObject(ygp_region, 6)
Map.addLayer(ygp_region, {}, 'Yunnan-Guizhou Plateau', True, 0.2)
Map.addLayer(img_col, {'bands': bands_vis, 'min': 0, 'max': 2000}, 'Landsat 7', True, 1)
Map.addLayer(region, {'color': 'red'}, 'search region', True, 0.5)
Map.addLayer(scene_sel, {'bands': bands_vis, 'min': 0, 'max': 2500}, 'selected scene', True, 1)
Map  



Map(center=[25.535259681661127, 102.94415803872928], controls=(WidgetControl(options=['position', 'transparent…